# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bushrahaider04/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**My Lane: Refresh / Content Opportunity Scoring**

I'm choosing Lane 2 because it has the clearest business value and the strongest existing foundation to build from. The starter pipeline already shows a working example that achieves Precision@50 = 0.740 with a random forest model, meaning it correctly identifies 37 out of the top 50 pages that need review. I can improve on this by moving beyond the simple "current decline" label to predict future decline 30 days ahead, using time-series features from the full 78.8M-row warehouse. This directly helps content teams prioritize their limited review capacity on pages most likely to lose traffic, preventing revenue loss before it happens.

## 2. The question: decision, action, cost of a wrong call

**The Decision:**
Content managers decide which pages to audit and potentially refresh each week. Currently, these decisions are reactive (fixing pages already in decline) or based on intuition.

**The Action:**
My model will produce a ranked queue of pages most likely to decline in the next 30 days. A content manager will:
1. Open the ranked list each week
2. Review the top 20-50 pages
3. For each page, inspect the reason codes explaining why it was flagged
4. Decide: refresh content, update metadata, or monitor

**Cost of a Wrong Call:**

*False Positive (flagging a page that won't decline):*
- Wastes 30 minutes of reviewer time per page
- 20 false positives × 30 min = 10 hours wasted weekly
- Reviewers lose trust in the model

*False Negative (missing a page that will decline):*
- Traffic drops 30-50% over 3 months
- If each page drives $500/month in conversions, losing 10 pages = $5,000/month lost
- Recovery becomes harder after 60+ days of decline

*Cost of doing nothing:*
- Continue reacting to declines instead of preventing them
- Competitors gain rankings while we wait for problems to become obvious

## 3. Quick look at the data (2-3 real numbers)

In [ ]:
# ============================================
# SECTION 3: QUICK LOOK AT THE DATA
# SAFE VERSION - No hardcoded token!
# ============================================

# Install huggingface
!pip install huggingface_hub -q

# Import libraries
from huggingface_hub import login, hf_hub_download
import pandas as pd
import numpy as np

# ============================================
# SECURE LOGIN - Enter your token when prompted
# ============================================

# This will securely prompt you for your token
# Your token will NOT be stored in the notebook
print("🔑 Please enter your Hugging Face token when prompted")
print("   (Get it from: https://huggingface.co/settings/tokens)")
login()

print("✅ Logged in successfully!")

# ============================================
# DOWNLOAD THE DATA
# ============================================

print("\n📥 Downloading fact_content_daily_performance_sample...")
fact_sample = pd.read_parquet(
    hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename="fact_content_daily_performance_sample.parquet",
        repo_type="dataset"
    )
)

print(f"✅ Loaded {len(fact_sample):,} rows")
print(f"📊 Columns: {fact_sample.columns.tolist()}")
print("\nFirst 5 rows:")
fact_sample.head()

print("\n📥 Downloading dim_content...")
dim_content = pd.read_parquet(
    hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename="dim_content.parquet",
        repo_type="dataset"
    )
)
print(f"✅ dim_content: {len(dim_content):,} rows")

print("\n📥 Downloading dim_clients...")
dim_clients = pd.read_parquet(
    hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        filename="dim_clients.parquet",
        repo_type="dataset"
    )
)
print(f"✅ dim_clients: {len(dim_clients):,} rows")

# ============================================
# ANALYZE THE DATA
# ============================================

print("\n" + "="*60)
print("📊 WAREHOUSE SAMPLE ANALYSIS")
print("="*60)
print(f"Total rows in sample: {len(fact_sample):,}")
print(f"Date range: {fact_sample['report_date'].min()} to {fact_sample['report_date'].max()}")
print(f"Unique content items: {fact_sample['content_hash_id'].nunique():,}")
print(f"Unique clients: {fact_sample['client_hash_id'].nunique():,}")

# Traffic metrics
total_impressions = fact_sample['gsc_impressions'].sum()
total_clicks = fact_sample['gsc_clicks'].sum()
total_sessions = fact_sample['ga4_sessions'].sum()
total_ai_sessions = fact_sample['sessions_ai'].sum()

print(f"\n📈 TRAFFIC METRICS:")
print(f"Total GSC impressions: {total_impressions:,.0f}")
print(f"Total GSC clicks: {total_clicks:,.0f}")
print(f"Total GA4 sessions: {total_sessions:,.0f}")
print(f"Total AI referral sessions: {total_ai_sessions:,.0f}")

# Pages with meaningful traffic
content_impressions = fact_sample.groupby('content_hash_id')['gsc_impressions'].sum()
pages_with_traffic = (content_impressions > 0).sum()
pages_with_meaningful_traffic = (content_impressions >= 500).sum()

print(f"\n📄 CONTENT PERFORMANCE:")
print(f"Pages with any impressions: {pages_with_traffic:,}")
print(f"Pages with >=500 impressions (meaningful traffic): {pages_with_meaningful_traffic:,}")
print(f"→ This is {pages_with_meaningful_traffic/pages_with_traffic*100:.1f}% of traffic-generating pages")

# CTR and position
ctr = (total_clicks / total_impressions * 100) if total_impressions > 0 else 0
avg_position = fact_sample[fact_sample['gsc_avg_position'] > 0]['gsc_avg_position'].mean()

print(f"\n🎯 PERFORMANCE METRICS:")
print(f"Overall CTR: {ctr:.2f}%")
print(f"Average search position: {avg_position:.1f}")

# Engagement
if 'ga4_engaged_sessions' in fact_sample.columns:
    engagement_rate = (fact_sample['ga4_engaged_sessions'].sum() / fact_sample['ga4_sessions'].sum() * 100) if fact_sample['ga4_sessions'].sum() > 0 else 0
    print(f"Engagement rate: {engagement_rate:.1f}%")

# Top pages
print(f"\n📋 TOP 10 PAGES BY IMPRESSIONS:")
top_pages = fact_sample.groupby('content_hash_id')['gsc_impressions'].sum().sort_values(ascending=False).head(10)
for idx, (page, impressions) in enumerate(top_pages.items(), 1):
    print(f"  {idx}. {page[:25]}... : {impressions:,.0f} impressions")

# Summary
print("\n" + "="*60)
print("🎯 KEY NUMBERS FOR YOUR WRITE-UP")
print("="*60)
print(f"1. Total content items: {len(dim_content):,}")
print(f"2. Pages with meaningful traffic (>=500 impressions): {pages_with_meaningful_traffic:,}")
print(f"3. Total GSC impressions: {total_impressions:,.0f}")
print(f"4. Total GSC clicks: {total_clicks:,.0f}")
print(f"5. Overall CTR: {ctr:.2f}%")
print(f"6. Average search position: {avg_position:.1f}")
print(f"7. Full warehouse rows: 81,769,613")
print(f"8. Monthly sample rows: {len(fact_sample):,}")
print(f"9. Unique clients: {fact_sample['client_hash_id'].nunique():,}")
print(f"10. Unique content items in sample: {fact_sample['content_hash_id'].nunique():,}")
print("="*60)

print("\n✅ ANALYSIS COMPLETE!")
print("\n💡 KEY INSIGHTS FOR YOUR NOTEBOOK:")
print(f"   - {pages_with_meaningful_traffic:,} pages need review")
print(f"   - {total_impressions:,.0f} total impressions")
print(f"   - Average position: {avg_position:.1f}")
print(f"   - CTR: {ctr:.2f}%")

### What These Numbers Tell Me

I loaded the warehouse sample (June 2026) from Hugging Face and found these key numbers:

**Number 1 - Scale of the Problem:**

| Metric | Value |
|--------|-------|
| Total content items | 519,606 |
| Pages with any impressions | 208,636 |
| Pages with meaningful traffic (>=500 impressions) | 52,766 |
| Percentage of traffic-generating pages with meaningful traffic | 25.3% |

**Why this matters:** With **52,766** pages generating meaningful traffic, there are thousands of pages at risk of decline. Even identifying 10-20% of these for early intervention could preserve significant traffic and revenue.

**Number 2 - Traffic and Engagement Metrics:**

| Metric | Value |
|--------|-------|
| Total GSC impressions | 216,194,872 |
| Total GSC clicks | 1,209,117 |
| Overall CTR | 0.56% |
| Average search position | 21.7 |
| Total GA4 sessions | 2,759,763 |
| Total AI referral sessions | 32,762 |

**Why this matters:** The average page ranks at position **21.7** - well below the first page. There's a huge opportunity to identify pages that could move up with content improvements. The **0.56% CTR** suggests many pages could benefit from better titles and meta descriptions.

**Number 3 - Data Depth for 7 Weeks:**

| Metric | Value |
|--------|-------|
| Full warehouse rows | 81,769,613 |
| Sample (June 2026) rows | 11,694,072 |
| Date range | June 1-30, 2026 |
| Unique clients | 65 |
| Unique content items in sample | 409,205 |

**Why this matters:** The full warehouse has **81.8M rows** across 18 months of history. This gives me enough data to build time-series features (trends, seasonality, momentum) that the starter model doesn't use. With **65 clients** and **409K content items**, I can build robust, generalizable models.

### Why This is Worth 7 Weeks

| Reason | Detail |
|--------|--------|
| **Scale** | 52,766 pages with meaningful traffic, 216M impressions - even small improvements matter |
| **Room for improvement** | Average position 21.7 and 0.56% CTR show many pages underperform |
| **Rich data** | 81.8M rows over 18 months allows time-series modeling |
| **Business impact** | Starter Precision@50 is 0.740 (37/50 correct). Improving to 0.800 means 3 more correct pages per week |
| **Proactive vs reactive** | Predict decline 30 days ahead instead of reacting to it |

### Key Numbers for Your Notebook

| # | Metric | Value |
|---|--------|-------|
| 1 | Pages with meaningful traffic | 52,766 |
| 2 | Total GSC impressions | 216,194,872 |
| 3 | Average search position | 21.7 |
| 4 | Overall CTR | 0.56% |
| 5 | Full warehouse rows | 81,769,613 |
| 6 | Monthly sample rows | 11,694,072 |
| 7 | Unique clients | 65 |
| 8 | Unique content items in sample | 409,205 |

## 4. Careful words: what I can and can't claim

### What I CAN claim (with evidence)

1. **Observational relationships:** "We observed that pages with declining impressions over 30 days and low CTR were more likely to have higher content age and lower engagement rates. In our June 2026 sample of 11.7M records, the average search position was 21.7 and overall CTR was 0.56%."

2. **Ranking improvement:** "Our model achieves Precision@50 = X.XXX, beating the baseline of 0.740 on held-out test data. The baseline correctly identifies 37 of the top 50 pages; our model identifies more."

3. **Actionable prioritization:** "Based on our model, these are the top 50 pages a content manager should review first from 52,766 pages with meaningful traffic."

4. **Correlation with future outcomes:** "Higher model scores are associated with a higher probability of future traffic decline. Pages scoring above 0.8 have a higher chance of losing 20%+ impressions in the next 30 days."

5. **Feature importance:** "The most important predictors in our model were position trend, CTR, and content age."

### What I CANNOT claim

1. **Causation:** ❌ "Refreshing these pages will cause recovery." 
   - I don't have experimental data (A/B tests) to prove causation

2. **Google's algorithm:** ❌ "We've discovered how Google ranks content."
   - We're observing correlations, not reverse-engineering algorithms

3. **Universal truth:** ❌ "These patterns apply to all content everywhere."
   - My findings are specific to 65 clients in the June 2026 sample

4. **Perfect prediction:** ❌ "This model can perfectly predict which pages will decline."
   - Even good models have false positives and negatives

5. **Content quality:** ❌ "Low scores mean the content is 'bad'."
   - We're measuring performance signals, not inherent content quality

### Why this is NOT just "train a model"

This is NOT just "train a model" because:

1. **Framing drives everything:** The research question determines what data I use, how I engineer features, what validation I choose, and how I interpret results

2. **Human judgment matters:** The output is decision support, not automation. Content managers still make the final call

3. **Reason codes are as important as scores:** A black-box score without explanation is useless

4. **The action defines success:** My work isn't done until someone can ACT on it

5. **Business context shapes design:** I choose Precision@50 because reviewers can only handle ~50 pages per week

### My honest limitations

**Data limitations:**
- Observational data, not experimental
- 65 clients in the sample (not all 70 in full warehouse)
- Only one month (June 2026) in the sample
- No actual refresh outcomes in the data

**Methodological limitations:**
- My label definition is one choice among many
- Different thresholds give different results
- Need to validate on truly held-out time periods
- Only 25.3% of traffic-generating pages have meaningful traffic (>=500 impressions)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Key Takeaways

**My lane:** Lane 2 - Refresh / Content Opportunity Scoring

**My question:** Can we predict which pages will decline in the next 30 days to prioritize content review?

**Decision:** Which pages to review first

**Action:** Content manager reviews top 50 pages weekly with reason codes

**Cost of wrong:** Wasted reviewer time OR lost traffic/revenue

### Real Numbers from Data

| Metric | Value |
|--------|-------|
| Total content items | 519,606 |
| Pages with meaningful traffic (>=500 impressions) | 52,766 |
| Total GSC impressions | 216,194,872 |
| Total GSC clicks | 1,209,117 |
| Overall CTR | 0.56% |
| Average search position | 21.7 |
| Full warehouse rows | 81,769,613 |
| Monthly sample rows | 11,694,072 |
| Unique clients | 65 |
| Unique content items in sample | 409,205 |

### Why NOT Just Train a Model

The question, human judgment, reason codes, and actionable output matter as much as the model.

**Success metric:** Beat starter Precision@50 (0.740) with a forward-looking 30-day prediction model

**I can change lanes until Week 4** - but Lane 2 gives me a clear, practical starting path.

### Next Steps

1. ✅ Research question framed with real numbers
2. ✅ Data loaded: 11,694,072 rows
3. ✅ Key metrics identified: 52,766 meaningful pages, 216M impressions
4. ⏭️ Run starter pipeline
5. ⏭️ Build improved model with time-series features
6. ⏭️ Validate and refine
7. ⏭️ Deploy paper